# RF-ADC state after overlay load: load-to-load variability

`adc_calibration.ipynb` runs 5 and 6 showed that the state after loading `ALS_LLRF_ZCU208` is not reproducible:
in some loads the loopback carrier is ~4 dB lower (-33 vs -28.8 dBFS) and 60 s of CW training reaches only
-49 to -69 dBc image level instead of -80 to -85 dBc.

This notebook loads the overlay `N_LOADS` times. For each load it:

1. freezes ADC calibration immediately (startup values, no test signal),
2. records clock and converter state (CLK104 lock, MTS offsets, ADC DSA, DAC output current, OCB2, startup GCB/TSCB),
3. turns on the CW probe and measures carrier level and the interleaving image at 6000 MHz − f_in,
4. trains 60 s on the probe, freezes, and measures again.

The summary table is meant to show which recorded quantity separates good loads from bad ones.

Setup as in `adc_calibration.ipynb`: DAC-to-ADC loopback into `ADC_CH`. With `USE_DAC = False`, feed `ADC_CH`
from an external generator instead (CW at `F_CW_MHZ`, about -30 dBFS): if the carrier level still splits into two
groups, the cause is on the ADC side. Restart the kernel before running.
About `N_LOADS` × 2 minutes.

In [1]:
%cd /home/xilinx/jupyter_notebooks/pynq_llrf/

/home/xilinx/jupyter_notebooks/pynq_llrf


In [2]:
import time

import numpy as np
import pandas as pd

from mimo_mts.overlay import MimoMtsOverlay

CONFIG = "ALS_LLRF_ZCU208"
ADC_CH = 0
F_CW_MHZ = 2973.0
N_LOADS = 6
TRAIN_S = 60
N_AVG = 4
TRIG_WAIT_S = 0.6
USE_DAC = True  # False: CW at F_CW_MHZ from an external signal generator into ADC_CH

## Helpers

In [8]:
ol = None


def load():
    """Free the previous overlay, load a new one and freeze ADC calibration straight away."""
    global ol, cal, fs_hz, fs_cap_hz
    if ol is not None:
        ol.free()
    ol = MimoMtsOverlay(config=CONFIG)
    cal = ol.adc_cal
    cal.freeze()
    fs_hz = ol.adc_sampling_rate
    fs_cap_hz = fs_hz / ol.ol_info["rfdc"]["adc_decimation_factor"]


def enable_cw(f_mhz=F_CW_MHZ):
    """Trigger setup, then the CW tone from the DAC (USE_DAC) or a muted DAC (external generator)."""
    rm = ol.rf_control.register_map
    rm.trig_sel, rm.trig_period, rm.trig_delay, rm.trig_divide = 0, 125e6, 0, 1
    rm.dac_enable, rm.pulse_length = 1, 0xFFF
    ol.dac_player[:] = np.zeros(ol.dac_player.size, dtype=np.int16)
    rm.amp_loop_setpoint = 32767 / ol.rf_control.tx_gain if USE_DAC else 0
    rm.phs_loop_setpoint = 0
    ol.dac_mixer_nco_freq_mhz = -f_mhz  # DAC 2nd Nyquist zone: NCO at -f gives a tone at +f


_last_capture = None


def capture(ch=ADC_CH, timeout_s=60):
    """One fresh capture: waits while the buffer is all zeros (trigger delay after load) or unchanged (stale)."""
    global _last_capture
    t0 = time.monotonic()
    while True:
        time.sleep(TRIG_WAIT_S)
        i, q = ol.capture_adc_iq_buf()
        x = i[ch].astype(np.float64) + 1j * q[ch].astype(np.float64)
        fresh = _last_capture is None or not np.array_equal(x, _last_capture)
        if np.any(x) and fresh:
            _last_capture = x
            return x
        if time.monotonic() - t0 > timeout_s:
            state = "all zeros" if not np.any(x) else "not updating (stale)"
            raise RuntimeError(f"ADC{ch} capture {state} after {timeout_s} s")


def levels(ch=ADC_CH):
    """Carrier (dBFS) and interleaving image (dBc) from N_AVG averaged spectra."""
    psd = 0
    for _ in range(N_AVG):
        x = capture(ch)
        win = np.blackman(len(x))
        psd = psd + np.abs(np.fft.fftshift(np.fft.fft(x * win)) / win.sum()) ** 2
    dbfs = 10 * np.log10(psd / N_AVG / 32768.0**2 + 1e-30)
    f = np.fft.fftshift(np.fft.fftfreq(len(x), 1 / fs_cap_hz)) / 1e6 + ol.adc_mixer_nco_freq_mhz

    def peak(f0, half_width=3):
        k = int(np.argmin(np.abs(f - f0)))
        return dbfs[max(0, k - half_width) : k + half_width + 1].max()

    carrier = peak(F_CW_MHZ)
    image = peak(1.5 * fs_hz / 1e6 - F_CW_MHZ)  # Fs/2 - f_in image, 2nd Nyquist zone
    return round(carrier, 1), round(image - carrier, 1)


def max_abs_slice(name, ch=ADC_CH):
    """Largest |coefficient| over the sub-ADC slices: GCB 12-bit, TSCB 9-bit signed."""
    bits, n = {"GCB": (12, 4), "TSCB": (9, 8)}[name]
    mask, sign = (1 << bits) - 1, 1 << (bits - 1)
    vals = []
    for v in cal.read(ch, [name])[ch][name][:n]:
        for s in (v & mask, (v >> 16) & mask):
            vals.append(s - (1 << bits) if s & sign else s)
    return int(np.abs(vals).max())


def safe(fn):
    try:
        return fn()
    except Exception as e:  # record, do not stop the loop
        return f"err: {type(e).__name__}"


def clk_regs(verifier, addrs):
    verifier.mux_select()
    return {a: verifier.read_reg(a) for a in addrs}


def lmx_state(verifier):
    r = clk_regs(verifier, (110, 111, 112))
    return {
        "LD_VTUNE": (r[110] >> 9) & 3,
        "VCO_SEL": (r[110] >> 5) & 7,
        "VCO_CAPCTRL": r[111] & 0xFF,
        "VCO_DACISET": r[112] & 0x1FF,
    }


def state_info():
    adc = cal.block(ADC_CH)
    clk = ol.clk104
    return {
        "clk104 locked": safe(lambda: bool(clk.is_locked())),
        "LMX ADC": safe(lambda: lmx_state(clk.v_lmx_adc)),
        "LMX DAC": safe(lambda: lmx_state(clk.v_lmx_dac)),
        # LMK04828 read-back: 0x183 lock/LOS flags, 0x184-0x185 PLL1 DAC value, 0x188 holdover status
        "LMK rb 0x183-0x188": safe(
            lambda: [hex(v) for v in clk_regs(clk.v_lmk, (0x183, 0x184, 0x185, 0x188)).values()]
        ),
        "MTS ADC offsets": safe(lambda: [int(ol.rfdc.mts_adc_config.Offset[t]) for t in range(4)]),
        "MTS DAC offsets": safe(lambda: [int(ol.rfdc.mts_dac_config.Offset[t]) for t in range(4)]),
        "ADC DSA": safe(lambda: dict(adc.DSA)),
        "OCB2[0]": safe(lambda: cal.read(ADC_CH, ["OCB2"])[ADC_CH]["OCB2"][0]),
        "startup |GCB| max": safe(lambda: max_abs_slice("GCB")),
        "startup |TSCB| max": safe(lambda: max_abs_slice("TSCB")),
    }

## Reload loop

In [4]:
rows = []
for k in range(N_LOADS):
    t0 = time.monotonic()
    load()
    row = {"load": k, **state_info()}
    enable_cw()
    row["carrier dBFS"], row["image startup dBc"] = levels()
    cal.unfreeze()
    time.sleep(TRAIN_S)
    cal.freeze()
    row["carrier after training dBFS"], row["image trained dBc"] = levels()
    row["trained |TSCB| max"] = max_abs_slice("TSCB")
    row["load + test s"] = round(time.monotonic() - t0)
    rows.append(row)
    print(row)

LMK04828: Changing Reg 0x15f from 0x3e to 0x3b. Fix the .tcs!


Powering up and initializing analog frontend ...
{'load': 0, 'clk104 locked': True, 'LMX ADC': {'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 81, 'VCO_DACISET': 300}, 'LMX DAC': {'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 74, 'VCO_DACISET': 312}, 'LMK rb 0x183-0x188': ['0x6', '0x88', '0x0', '0x10'], 'MTS ADC offsets': [1, 1, 1, 1], 'MTS DAC offsets': [2, 2, 2, 2], 'ADC DSA': {'DisableRTS': 0, 'Attenuation': 0.0}, 'OCB2[0]': 5374001, 'startup |GCB| max': 1, 'startup |TSCB| max': 172, 'carrier dBFS': -32.9, 'image startup dBc': -38.0, 'carrier after training dBFS': -33.1, 'image trained dBc': -42.3, 'trained |TSCB| max': 94, 'load + test s': 98}


LMK04828: Changing Reg 0x15f from 0x3e to 0x3b. Fix the .tcs!


Powering up and initializing analog frontend ...
{'load': 1, 'clk104 locked': True, 'LMX ADC': {'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 81, 'VCO_DACISET': 300}, 'LMX DAC': {'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 73, 'VCO_DACISET': 312}, 'LMK rb 0x183-0x188': ['0x6', '0x88', '0x0', '0x10'], 'MTS ADC offsets': [1, 1, 1, 1], 'MTS DAC offsets': [2, 2, 2, 2], 'ADC DSA': {'DisableRTS': 0, 'Attenuation': 0.0}, 'OCB2[0]': 5374001, 'startup |GCB| max': 1, 'startup |TSCB| max': 162, 'carrier dBFS': -28.7, 'image startup dBc': -47.4, 'carrier after training dBFS': -28.7, 'image trained dBc': -81.1, 'trained |TSCB| max': 109, 'load + test s': 96}


LMK04828: Changing Reg 0x15f from 0x3e to 0x3b. Fix the .tcs!


Powering up and initializing analog frontend ...
{'load': 2, 'clk104 locked': True, 'LMX ADC': {'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 82, 'VCO_DACISET': 299}, 'LMX DAC': {'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 73, 'VCO_DACISET': 312}, 'LMK rb 0x183-0x188': ['0x6', '0x88', '0x0', '0x10'], 'MTS ADC offsets': [1, 1, 1, 1], 'MTS DAC offsets': [2, 2, 2, 2], 'ADC DSA': {'DisableRTS': 0, 'Attenuation': 0.0}, 'OCB2[0]': 5374001, 'startup |GCB| max': 1, 'startup |TSCB| max': 220, 'carrier dBFS': -32.1, 'image startup dBc': -40.4, 'carrier after training dBFS': -32.0, 'image trained dBc': -54.4, 'trained |TSCB| max': 248, 'load + test s': 95}


... (2 outputs left out)

LMK04828: Changing Reg 0x15f from 0x3e to 0x3b. Fix the .tcs!


Powering up and initializing analog frontend ...
{'load': 4, 'clk104 locked': True, 'LMX ADC': {'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 81, 'VCO_DACISET': 299}, 'LMX DAC': {'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 73, 'VCO_DACISET': 312}, 'LMK rb 0x183-0x188': ['0x6', '0x88', '0x0', '0x10'], 'MTS ADC offsets': [1, 1, 1, 1], 'MTS DAC offsets': [2, 2, 2, 2], 'ADC DSA': {'DisableRTS': 0, 'Attenuation': 0.0}, 'OCB2[0]': 5308464, 'startup |GCB| max': 1, 'startup |TSCB| max': 219, 'carrier dBFS': -32.4, 'image startup dBc': -40.3, 'carrier after training dBFS': -32.6, 'image trained dBc': -48.6, 'trained |TSCB| max': 189, 'load + test s': 95}


LMK04828: Changing Reg 0x15f from 0x3e to 0x3b. Fix the .tcs!


Powering up and initializing analog frontend ...
{'load': 5, 'clk104 locked': True, 'LMX ADC': {'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 81, 'VCO_DACISET': 300}, 'LMX DAC': {'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 74, 'VCO_DACISET': 312}, 'LMK rb 0x183-0x188': ['0x6', '0x88', '0x0', '0x10'], 'MTS ADC offsets': [1, 1, 1, 1], 'MTS DAC offsets': [2, 2, 2, 2], 'ADC DSA': {'DisableRTS': 0, 'Attenuation': 0.0}, 'OCB2[0]': 5374002, 'startup |GCB| max': 1, 'startup |TSCB| max': 224, 'carrier dBFS': -32.8, 'image startup dBc': -38.4, 'carrier after training dBFS': -33.1, 'image trained dBc': -45.0, 'trained |TSCB| max': 122, 'load + test s': 96}


## Summary

Sorted by carrier level. Look for a column that changes together with the carrier level and the trained image.

In [5]:
df = pd.DataFrame(rows).set_index("load")
df.sort_values("carrier dBFS")

,clk104 locked,LMX ADC,LMX DAC,LMK rb 0x183-0x188,MTS ADC offsets,MTS DAC offsets,ADC DSA,OCB2[0],startup |GCB| max,startup |TSCB| max,carrier dBFS,image startup dBc,carrier after training dBFS,image trained dBc,trained |TSCB| max,load + test s
load,,,,,,,,,,,,,,,,
0,True,"{'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 8...","{'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 7...","[0x6, 0x88, 0x0, 0x10]","[1, 1, 1, 1]","[2, 2, 2, 2]","{'DisableRTS': 0, 'Attenuation': 0.0}",5374001,1,172,-32.9,-38.0,-33.1,-42.3,94,98
3,True,"{'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 8...","{'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 7...","[0x6, 0x88, 0x0, 0x10]","[1, 1, 1, 1]","[2, 2, 2, 2]","{'DisableRTS': 0, 'Attenuation': 0.0}",5505075,1,176,-32.9,-37.8,-33.2,-46.0,142,96
5,True,"{'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 8...","{'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 7...","[0x6, 0x88, 0x0, 0x10]","[1, 1, 1, 1]","[2, 2, 2, 2]","{'DisableRTS': 0, 'Attenuation': 0.0}",5374002,1,224,-32.8,-38.4,-33.1,-45.0,122,96
4,True,"{'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 8...","{'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 7...","[0x6, 0x88, 0x0, 0x10]","[1, 1, 1, 1]","[2, 2, 2, 2]","{'DisableRTS': 0, 'Attenuation': 0.0}",5308464,1,219,-32.4,-40.3,-32.6,-48.6,189,95
2,True,"{'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 8...","{'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 7...","[0x6, 0x88, 0x0, 0x10]","[1, 1, 1, 1]","[2, 2, 2, 2]","{'DisableRTS': 0, 'Attenuation': 0.0}",5374001,1,220,-32.1,-40.4,-32.0,-54.4,248,95
1,True,"{'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 8...","{'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 7...","[0x6, 0x88, 0x0, 0x10]","[1, 1, 1, 1]","[2, 2, 2, 2]","{'DisableRTS': 0, 'Attenuation': 0.0}",5374001,1,162,-28.7,-47.4,-28.7,-81.1,109,96


## Tile reset test: DAC side or ADC side?

None of the clock-chain readings separates good from bad loads. A DAC sampling-clock duty-cycle error would put a
spur at Fs/2 + 1027 MHz = 3027 MHz, exactly where the ADC image is measured, and would also change the mix-mode
output level. This section loads the overlay once, freezes the ADC calibration, then resets **only the DAC tiles**
`N_RESETS` times, then **only the ADC tiles** `N_RESETS` times. After each reset the mixers and the CW tone are
re-applied and the carrier and the 3027 MHz level are measured.

* carrier flips between ~-28.8 and ~-33 dBFS across DAC resets → the bad state is on the DAC side;
* flips across ADC resets → ADC side.

MTS alignment is not restored after these resets; it is not needed for this measurement.

In [6]:
N_RESETS = 6


def reset_tiles(kind):
    """Reset the active DAC or ADC tiles only, then re-apply mixers and the CW tone."""
    tiles, mask = {
        "dac": (ol.rfdc.dac_tiles, ol.board.active_dac_tiles),
        "adc": (ol.rfdc.adc_tiles, ol.board.active_adc_tiles),
    }[kind]
    for t in range(4):
        if mask >> t & 1:
            tiles[t].Reset()
    ol._initialize_mixers()
    enable_cw()


load()
enable_cw()
carrier, image = levels()
reset_rows = [{"reset": "initial load", "carrier dBFS": carrier, "3027 MHz dBc": image}]
print(reset_rows[0])
for kind in ("dac", "adc"):
    for k in range(N_RESETS):
        reset_tiles(kind)
        carrier, image = levels()
        row = {"reset": f"{kind} #{k}", "carrier dBFS": carrier, "3027 MHz dBc": image}
        reset_rows.append(row)
        print(row, "frozen:", all(cal.is_frozen().values()))
pd.DataFrame(reset_rows).set_index("reset")

LMK04828: Changing Reg 0x15f from 0x3e to 0x3b. Fix the .tcs!


Powering up and initializing analog frontend ...
{'reset': 'initial load', 'carrier dBFS': -32.9, '3027 MHz dBc': -37.9}
{'reset': 'dac #0', 'carrier dBFS': -32.9, '3027 MHz dBc': -37.9} frozen: True
{'reset': 'dac #1', 'carrier dBFS': -32.9, '3027 MHz dBc': -38.0} frozen: True
{'reset': 'dac #2', 'carrier dBFS': -32.9, '3027 MHz dBc': -37.9} frozen: True
{'reset': 'dac #3', 'carrier dBFS': -32.9, '3027 MHz dBc': -37.9} frozen: True
{'reset': 'dac #4', 'carrier dBFS': -32.9, '3027 MHz dBc': -37.9} frozen: True
{'reset': 'dac #5', 'carrier dBFS': -32.9, '3027 MHz dBc': -38.0} frozen: True
{'reset': 'adc #0', 'carrier dBFS': -32.8, '3027 MHz dBc': 13.3} frozen: True
{'reset': 'adc #1', 'carrier dBFS': -19.6, '3027 MHz dBc': -11.6} frozen: True
{'reset': 'adc #2', 'carrier dBFS': -19.9, '3027 MHz dBc': -8.7} frozen: True
{'reset': 'adc #3', 'carrier dBFS': -21.5, '3027 MHz dBc': -1.8} frozen: True
{'reset': 'adc #4', 'carrier dBFS': -19.6, '3027 MHz dBc': -11.4} frozen: True
{'reset': 'ad

,carrier dBFS,3027 MHz dBc
reset,,
initial load,-32.9,-37.9
dac #0,-32.9,-37.9
dac #1,-32.9,-38.0
dac #2,-32.9,-37.9
dac #3,-32.9,-37.9
dac #4,-32.9,-37.9
dac #5,-32.9,-38.0
adc #0,-32.8,13.3
adc #1,-19.6,-11.6


### ADC restarts with MTS re-sync

DAC-only resets never changed the state, so the DAC side is not the cause. ADC-only resets without re-running
MTS and the FIFO setup scrambled the captured data (results above are invalid). Here each ADC restart uses
`ol.restart_adc_tiles()`, which re-runs RF-ADC startup and then MTS and the mixers (it resets the DAC tiles too,
shown above not to matter). `check_mts_latency()` raises if alignment failed.

* carrier switches between ~-28.8 and ~-33 dBFS across restarts → the bad state is set at RF-ADC startup, and
  restarting the ADC tiles until the load is good is a possible workaround;
* never switches → the state is fixed when the overlay loads (clock chip programming or PL), not by the ADC tile.

In [9]:
load()  # fresh overlay: the ADC-only resets above left the capture path stuck
enable_cw()
carrier, image = levels()
adc_rows = [{"restart": "initial load", "carrier dBFS": carrier, "3027 MHz dBc": image}]
print(adc_rows[0])
for k in range(N_RESETS):
    ol.restart_adc_tiles()
    ol.check_mts_latency()
    enable_cw()
    carrier, image = levels()
    row = {"restart": k, "carrier dBFS": carrier, "3027 MHz dBc": image}
    adc_rows.append(row)
    print(row, "frozen:", all(cal.is_frozen().values()))
pd.DataFrame(adc_rows).set_index("restart")

LMK04828: Changing Reg 0x15f from 0x3e to 0x3b. Fix the .tcs!


Powering up and initializing analog frontend ...
{'restart': 'initial load', 'carrier dBFS': -32.3, '3027 MHz dBc': -39.5}
{'restart': 0, 'carrier dBFS': -32.6, '3027 MHz dBc': -44.7} frozen: True
{'restart': 1, 'carrier dBFS': -33.9, '3027 MHz dBc': -54.6} frozen: True
{'restart': 2, 'carrier dBFS': -33.9, '3027 MHz dBc': -54.6} frozen: True
{'restart': 3, 'carrier dBFS': -33.9, '3027 MHz dBc': -54.7} frozen: True
{'restart': 4, 'carrier dBFS': -33.9, '3027 MHz dBc': -58.6} frozen: True
{'restart': 5, 'carrier dBFS': -33.9, '3027 MHz dBc': -57.4} frozen: True


,carrier dBFS,3027 MHz dBc
restart,,
initial load,-32.3,-39.5
0,-32.6,-44.7
1,-33.9,-54.6
2,-33.9,-54.6
3,-33.9,-54.7
4,-33.9,-58.6
5,-33.9,-57.4


## Clock re-programming test: is the state set by the CLK104?

Neither DAC nor ADC tile restarts change the carrier level, so the bad state is set upstream of the converters.
Here only the CLK104 (LMK04828 + both LMX2594) is re-programmed, without reloading the bitstream, followed by
`restart_adc_tiles()` (resets ADC and DAC tiles, re-runs MTS and the mixers) and a measurement.

* carrier switches between ~-28.8 and ~-33 dBFS across re-programmings → the state comes from clock-chip startup
  (LMK04828 in PLL mode is the main suspect; the `_DIST` configuration with the reference on CLKin1 is the fix to
  test). Re-programming until good would be a workaround.
* never switches → the cause is in the bitstream / PL configuration at load time.

ADC calibration stays frozen; each tile restart resets TSCB to near zero, so the image is only indicative here.

In [10]:
def reprogram_clocks():
    """Re-write the .tcs values to LMK04828 and both LMX2594, restore the read-back settings, return lock."""
    clk = ol.clk104
    clk.write_regs()
    # write_regs() restores the .tcs values; re-apply the SPI read-back settings the verifiers enabled
    for v, addrs in ((clk.v_lmk, (0x15F, 0x16E)), (clk.v_lmx_adc, (0,)), (clk.v_lmx_dac, (0,))):
        for a in addrs:
            if a in v.reg_map:
                v.write_reg(a, v.reg_map[a])
    time.sleep(1.0)  # PLL lock and LMX VCO calibration
    return bool(clk.is_locked())


load()
enable_cw()
carrier, image = levels()
clk_rows = [{"step": "initial load", "carrier dBFS": carrier, "3027 MHz dBc": image}]
print(clk_rows[0])
for k in range(N_RESETS):
    locked = reprogram_clocks()
    ol.restart_adc_tiles()
    ol.check_mts_latency()
    enable_cw()
    carrier, image = levels()
    row = {
        "step": f"reprogram #{k}",
        "clk104 locked": locked,
        "LMX ADC": safe(lambda: lmx_state(ol.clk104.v_lmx_adc)),
        "carrier dBFS": carrier,
        "3027 MHz dBc": image,
    }
    clk_rows.append(row)
    print(row)
pd.DataFrame(clk_rows).set_index("step")

LMK04828: Changing Reg 0x15f from 0x3e to 0x3b. Fix the .tcs!


Powering up and initializing analog frontend ...
{'step': 'initial load', 'carrier dBFS': -28.7, '3027 MHz dBc': -46.1}
{'step': 'reprogram #0', 'clk104 locked': True, 'LMX ADC': {'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 81, 'VCO_DACISET': 300}, 'carrier dBFS': -28.8, '3027 MHz dBc': -51.1}
{'step': 'reprogram #1', 'clk104 locked': True, 'LMX ADC': {'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 82, 'VCO_DACISET': 300}, 'carrier dBFS': -32.9, '3027 MHz dBc': -54.1}
{'step': 'reprogram #2', 'clk104 locked': True, 'LMX ADC': {'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 81, 'VCO_DACISET': 300}, 'carrier dBFS': -33.3, '3027 MHz dBc': -54.2}
{'step': 'reprogram #3', 'clk104 locked': True, 'LMX ADC': {'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 81, 'VCO_DACISET': 300}, 'carrier dBFS': -28.9, '3027 MHz dBc': -59.0}
{'step': 'reprogram #4', 'clk104 locked': True, 'LMX ADC': {'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 82, 'VCO_DACISET': 299}, 'carrier dBFS': -31.5, '3027 MHz dBc': -58.7}
{'ste

,carrier dBFS,3027 MHz dBc,clk104 locked,LMX ADC
step,,,,
initial load,-28.7,-46.1,NaN,NaN
reprogram #0,-28.8,-51.1,True,"{'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 8..."
reprogram #1,-32.9,-54.1,True,"{'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 8..."
reprogram #2,-33.3,-54.2,True,"{'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 8..."
reprogram #3,-28.9,-59.0,True,"{'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 8..."
reprogram #4,-31.5,-58.7,True,"{'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 8..."
reprogram #5,-28.8,-59.4,True,"{'LD_VTUNE': 2, 'VCO_SEL': 1, 'VCO_CAPCTRL': 8..."
